RNN - Erro dos pesos computados e usado somente durante a iteração

In [1]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import operator as op
import ipynbname
import math
import matplotlib.cm as cm
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Utils_RTLO import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE
FileName = ipynbname.name()

df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values
df = pd.read_csv(r'Dataset\Dataset_1_NCA_battery_SoH\CY25-05_1-#01.csv')
sig = df['soh'].values
    
def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler


In [175]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,mode='past'):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.ref = None
        self.act = 'tanh'
        self.flw = mode
        self.n = -1
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.08

        self.xPi = np.zeros(nI)
        self.hP, self.hU, self.hL = [0.1*np.ones(nR) for i in range(3)]

        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)
        #self.BS = np.random.randn(nR, nO)/nO**0.5
        
        self.yP, self.yR, self.yL, self.yU = [np.array([]) for i in range(4)]
        self.yP_hist = np.zeros(self.nI)

        self.εY, self.εM, self.εE, self.εR, self.ΣW = [0 for i in range(5)]
        self.εM_hist = np.array([])
        self.εR_hist = np.array([])

        self.wR_hist = []
        self.wI_hist = []
        self.wO_hist = []

        self.rR = 1e-9
        self.rP = 1e-10
        self.rL = 1e-11
        self.rU = 1e-12
        self.rRsum = 0

        self.rulR, self.rulP, self.rulL, self.rulU = [np.array([]) for i in range(4)]


    def PredSingle(self,x):

        u = np.dot(self.wR, self.hS) + np.dot(self.wI, x)
        h = self.hS + (-self.hS + Activation(u,self.act))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR,start=0,store=False,show=False):
        if self.flw != 'past': self.n = 0
        
        η1,η2,η3 = self.ηS      

        uS = self.wR @ self.hP + self.wI @ xP
        hP = self.hP*(1-1/self.τ) +Activation(uS,self.act)/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        self.pS = np.outer(dActivation(uS,self.act),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(uS,self.act),self.xPi)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.wR_hist.append(self.wR.flatten())
        self.wI_hist.append(self.wI.flatten())
        self.wO_hist.append(self.wO.flatten())

        self.hP = hP
        self.xPi = xP

        if self.k>=start:
            W = self.k**2
            ΣW = self.ΣW + W
            ΔY = np.abs((yR-yP)/yR)
            ΔM = np.linalg.norm(ΔY,ord=2)
            ΔR = np.abs((self.rR-self.rP)/(self.rR+1e-9))

            self.εY = ((self.εY*self.ΣW) + (W*ΔY[self.n]))/ΣW
            self.εM = ((self.εM*self.ΣW) + (W*ΔM))/ΣW
            self.εR = ((self.εR*self.ΣW) + (W*ΔR))/ΣW
            self.ΣW = ΣW

        if store:
            self.yR = np.append(self.yR,yR[0])
            if self.k>=start:
                self.εM_hist = np.append(self.εM_hist,self.εM)
                self.εR_hist = np.append(self.εR_hist,self.εR)

        self.k = self.k+1
        self.t = np.append(self.t,self.k + self.nI)
        self.yP_hist = np.delete(np.append(self.yP_hist,yP[0]),0)
        #self.ηS = self.ηS/(1 + self.decay*self.k)


    '''def PredRulIntr(self, x,lim=0.2,maxRul=110,store=False,show=False):
        #print('yH:',self.yP_hist)
        #print('eS: ',self.eS)

        for i,y in enumerate(self.yP_hist):
            if y != 0:
                self.eS2[i] = self.rls.predict(np.abs(y))
        #print('eS2:',self.eS2)

        xP,xL,xU =x.copy(), (x-(self.ρ*self.eS2)).copy(),(x+(self.ρ*self.eS2)).copy()    

        #xP,xL,xU =x.copy(), x.copy()*0.999, x.copy()*1.00

        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO

        wRp, wRn = np.maximum(0, self.wR), np.abs(np.minimum(0, self.wR))
        wIp, wIn = np.maximum(0, self.wI), np.abs(np.minimum(0, self.wI))
        wOp, wOn = np.maximum(0, self.wO), np.abs(np.minimum(0, self.wO))
        #hU,hL = np.maximum(0, self.hS), np.minimum(0, self.hS)
        hL,hP,hU = [self.hP.copy() for i in range(3)]

        while predict:
            uP = wR@hP + wI@xP
            uL = (wRp @ hL - wRn @ hU) + (wIp @ xL - wIn @ xU)
            uU = (wRp @ hU - wRn @ hL) + (wIp @ xU - wIn @ xL)

            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
    
            yP = (wO@hP)
            yL = (wOp @ hL - wOn @ hU)
            yU = (wOp @ hU - wOn @ hL)

            #if show: print(yP)
            yP = yP[0]
            yL = yL[0]
            yU = yU[0]

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)
            PredVals = [yL,yP,yU]

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)'''
    
    def PredRulIntr2(self, x,lim=0.2,maxRul=110,store=False,show=False):
        if self.flw != 'past': self.n = 0
        xP,xL,xU =x.copy(), x.copy(), x.copy()
        k = 1
        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO
        hP = self.hP.copy() 
        ep = 1+self.ρ

        wRU, wRL = np.maximum(ep*wR, wR/ep), np.minimum(ep*wR, wR/ep)
        wIU, wIL = np.maximum(ep*wI, wI/ep), np.minimum(ep*wI, wI/ep)
        wOU, wOL = np.maximum(ep*wO, wO/ep), np.minimum(ep*wO, wO/ep) 
        #hU ,  hL = np.maximum(ep*hP, hP/ep), np.minimum(ep*hP, hP/ep)
        #print(wIL[0], wI[0], wIU[0])
        #print(wIL[1], wI[1], wIU[1])
        #print(wIL[2], wI[2], wIU[2])
        #wRU, wRL = np.maximum(0, wRU), np.abs(np.minimum(0, wRL))
        #wIU, wIL = np.maximum(0, wIU), np.abs(np.minimum(0, wIL))
        #wOU, wOL = np.maximum(0, wOU), np.abs(np.minimum(0, wOL))
        #hU, hL   = np.maximum(0,  hU), np.abs(np.minimum(0, hL))
        #print(wIL[0], wI[0], wIU[0])
        #print(wIL[1], wI[1], wIU[1])
        #print(wIL[2], wI[2], wIU[2])
        #git

        hU ,  hL = self.hP.copy() , self.hP.copy() 

        '''ep=self.ρ

        wRU, wRL = np.maximum((1+ep)*wR, (1-ep)*wR), np.minimum((1+ep)*wR, (1-ep)*wR)
        wIU, wIL = np.maximum((1+ep)*wI, (1-ep)*wI), np.minimum((1+ep)*wI, (1-ep)*wI)
        wOU, wOL = np.maximum((1+ep)*wO, (1-ep)*wO), np.minimum((1+ep)*wO, (1-ep)*wO)
        hU, hL = np.maximum((1+ep)*hP, (1-ep)*hP), np.minimum((1+ep)*hP, (1-ep)*hP)'''
        while predict:
            #if show:
                #print(k,PredVals)
                #print(k,uL,uP,uU)
                #print(k,hL,hP,hU)
                #print(( wR @ hP) + ( wI @ xP))
                #print((wRU @ hL) - (wRL @ hU) + (wIU @ xL) - (wIL @ xU))
                #print((wRU @ hU) - (wRL @ hL) + (wIU @ xU) - (wIL @ xL))
                #print()
            uP = ( wR @ hP) + ( wI @ xP)
            uL = (wRU @ hL) - (wRL @ hU) + (wIU @ xL) - (wIL @ xU)
            uU = (wRU @ hU) - (wRL @ hL) + (wIU @ xU) - (wIL @ xL)

            #uU, uL = np.maximum(uU,uL), np.minimum(uU,uL)
            
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ

            #hU,hL= hP,hP
            #hU, hL = np.maximum(hU,hL), np.minimum(hU,hL)

            yP = ( wO @ hP)
            yL = (wOU @ hL) - (wOL @ hU)
            yU = (wOU @ hU) - (wOL @ hL)
            if show:   
                print(k,yL,yP,yU)
            #yU, yL = np.maximum(yU,yL), np.minimum(yU,yL)

            #print(hL,hP)
            PredVals = ([yL[self.n],yP[self.n],yU[self.n]])
            yL,yP,yU = PredVals

            #if show:
                #print(k,PredVals)
                #print(k,uL,uP,uU)
                #print(k,hL,hP,hU)

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
            k = k+1
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRul(self, x,maxRul=100,lim=0.2,store=False):
        if self.flw != 'past': self.n = 0
        xP = x.copy()
        rulP=0
        predict = True
        hP = self.hP.copy()
        while predict:
            uP = (self.wR @ hP) + (self.wI @ xP)
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            yP = (self.wO @ hP)[self.n]
            xP = np.delete(np.append(xP,yP),0)
            if store:
                if rulP==0:
                    self.yP = np.append(self.yP,yP)

            if predict: rulP = rulP+1
            if yP < lim: predict = False
            if rulP >= maxRul:
                break

        self.rR=self.ref-self.k
        self.rP = rulP

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulP = np.append(self.rulP,self.rP)
            



#Optimize parameters for minimize error of degradation prediction

In [ ]:
rates = [1/(10**i) for i in range(1,8)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 3, 3) 
    nR = trial.suggest_int('nR', 3, 3) 
    nO = trial.suggest_int('nO', 3, 3) 
    N1 = trial.suggest_categorical('N1', rates[:]) 
    N2 = trial.suggest_categorical('N2', rates[:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 40)    
    mode = trial.suggest_categorical('mode', ['past'])  
    cont=0
    if mode == 'past':
        if nO > nI:
            raise TrialPruned()
    X,Y = PrepareData(sig,nI,nO,mode)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.PredRul(X[i],maxRul=len(sig)-nI,lim=0.75,store=True)
        rnn.fit(X[i],Y[i],store=True)

        if i == int((len(sig)-nI)/2):
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()
            if np.mean(rnn.rulP)>np.mean(rnn.rulR)*1.25:
                raise TrialPruned()
            if np.mean(rnn.rulP)<np.mean(rnn.rulR)*0.25:
                raise TrialPruned()
        if i>=30:
            if rnn.rulP[i]==rnn.rulP[i-1]:
                cont = cont+1
            if cont> 20:
                raise TrialPruned()
            
    #return rnn.εY
    return rnn.εM + rnn.εR

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="minimize",
    sampler=SelSampler(mode='random'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=20000)
params = list(study.best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

Erro_M = 0.132222, Erro_R = 0.145739 Parâmetros: {'nI': 18, 'nR': 41, 'nO': 16, 'N1': 0.1, 'N2': 0.001, 'N3': 0.001, 'τ': 21}\
Erro_M = 0.117815, Erro_R = 0.147648 Parâmetros: {'nI': 3, 'nR': 25, 'nO': 6, 'N1': 0.1, 'N2': 0.01, 'N3': 1e-06, 'τ': 18}\
Erro_M = 0.106091, Erro_R = 0.144622 Parâmetros: {'nI': 18, 'nR': 31, 'nO': 1, 'N1': 1e-07, 'N2': 0.001, 'N3': 1e-05, 'τ': 21}\
Erro_M = 0.063852, Erro_R = 0.121791 Parâmetros: {'nI': 15, 'nR': 34, 'nO': 3, 'N1': 0.1, 'N2': 0.1, 'N3': 0.0001, 'τ': 25}\
Erro_M = 0.063259, Erro_R = 0.194122 Parâmetros: {'nI': 7, 'nR': 31, 'nO': 6, 'N1': 0.1, 'N2': 1e-05, 'N3': 1e-07, 'τ': 20}\
Erro_M = 0.049165, Erro_R = 0.169135 Parâmetros: {'nI': 22, 'nR': 48, 'nO': 6, 'N1': 0.1, 'N2': 0.001, 'N3': 0.0001, 'τ': 12}\





In [180]:
rates = [1/(10**i) for i in range(1,9)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 2, 2) 
    nR = trial.suggest_int('nR', 2, 5) 
    nO = trial.suggest_int('nO', 2, 2) 
    N1 = trial.suggest_categorical('N1', rates[:]) 
    N2 = trial.suggest_categorical('N2', rates[:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 40)    
    mode = trial.suggest_categorical('mode', ['past'])  
    if mode == 'past':
        if nO > nI:
            raise TrialPruned()
    X,Y = PrepareData(sig,nI,nO,mode)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.PredRul(X[i],maxRul=len(sig),lim=0.75,store=True)
        rnn.fit(X[i],Y[i],start=0,store=True)
        #rnn.act='relu'

        if i == int((len(sig)-nI)/2):
            #if np.mean(rnn.εM_hist)>1:
            #    raise TrialPruned()
            if np.mean(rnn.rulP)>np.mean(rnn.rulR)*1.3:
                raise TrialPruned()
            if np.mean(rnn.rulP)<np.mean(rnn.rulR)*0.3:
                raise TrialPruned()
        '''if i>=30:
            if rnn.rulP[i]==rnn.rulP[i-1]:
                cont = cont+1
            if cont> 20:
                raise TrialPruned()'''
            
    return rnn.εM, rnn.εR

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    directions=["minimize", "minimize"],
    sampler=SelSampler(mode='random'),
    #pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=5000)

best_trials = study.best_trials

print(f"Encontrados {len(best_trials)} modelos na Fronteira de Pareto:")

for i, trial in enumerate(best_trials):
    print(f"Erro_M = {trial.values[0]:.6f}, Erro_R = {trial.values[1]:.6f}",
          f"Parâmetros: {trial.params}")
# Se você quiser apenas os parâmetros do PRIMEIRO modelo da fronteira para testar:
first_best_params = best_trials[0].params
params = list(trial.params.values())


[I 2026-04-29 13:32:02,090] A new study created in memory with name: no-name-534cb5b6-fe78-4c5c-b460-cc90c2991159
[I 2026-04-29 13:32:02,104] Trial 0 pruned. 
[I 2026-04-29 13:32:02,110] Trial 1 pruned. 
[I 2026-04-29 13:32:02,114] Trial 2 pruned. 
[I 2026-04-29 13:32:02,123] Trial 3 pruned. 
[I 2026-04-29 13:32:02,127] Trial 4 pruned. 
[I 2026-04-29 13:32:02,133] Trial 5 pruned. 
[I 2026-04-29 13:32:02,271] Trial 6 finished with values: [0.06402422929223758, 8.638998793290071] and parameters: {'nI': 2, 'nR': 5, 'nO': 2, 'N1': 0.1, 'N2': 1e-06, 'N3': 1e-06, 'τ': 17, 'mode': 'past'}.
[I 2026-04-29 13:32:02,275] Trial 7 pruned. 
[I 2026-04-29 13:32:02,283] Trial 8 pruned. 
[I 2026-04-29 13:32:02,293] Trial 9 pruned. 
[I 2026-04-29 13:32:02,299] Trial 10 pruned. 
[I 2026-04-29 13:32:02,305] Trial 11 pruned. 
[I 2026-04-29 13:32:02,309] Trial 12 pruned. 
[I 2026-04-29 13:32:02,318] Trial 13 pruned. 
[I 2026-04-29 13:32:02,321] Trial 14 pruned. 
[I 2026-04-29 13:32:02,330] Trial 15 pruned. 

Encontrados 11 modelos na Fronteira de Pareto:
Erro_M = 0.016627, Erro_R = 0.823793 Parâmetros: {'nI': 2, 'nR': 2, 'nO': 2, 'N1': 0.1, 'N2': 1e-06, 'N3': 0.1, 'τ': 8, 'mode': 'past'}
Erro_M = 0.011625, Erro_R = 1.837802 Parâmetros: {'nI': 2, 'nR': 5, 'nO': 2, 'N1': 0.1, 'N2': 1e-07, 'N3': 0.1, 'τ': 3, 'mode': 'past'}
Erro_M = 0.016307, Erro_R = 0.829839 Parâmetros: {'nI': 2, 'nR': 2, 'nO': 2, 'N1': 0.1, 'N2': 0.01, 'N3': 0.1, 'τ': 8, 'mode': 'past'}
Erro_M = 0.028845, Erro_R = 0.359847 Parâmetros: {'nI': 2, 'nR': 2, 'nO': 2, 'N1': 0.1, 'N2': 0.1, 'N3': 0.1, 'τ': 20, 'mode': 'past'}
Erro_M = 0.009477, Erro_R = 3.797831 Parâmetros: {'nI': 2, 'nR': 2, 'nO': 2, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.1, 'τ': 1, 'mode': 'past'}
Erro_M = 0.010531, Erro_R = 2.246241 Parâmetros: {'nI': 2, 'nR': 5, 'nO': 2, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.1, 'τ': 2, 'mode': 'past'}
Erro_M = 0.012919, Erro_R = 1.671887 Parâmetros: {'nI': 2, 'nR': 5, 'nO': 2, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.1, 'τ': 4, 'mode': 'past'}
Err

Erro_M = 0.013465, Erro_R = 0.101836 Parâmetros: {'nI': 15, 'nR': 37, 'nO': 18, 'N1': 0.1, 'N2': 0.001, 'N3': 0.01, 'τ': 18, 'mode': 'ahead'}\
 Erro_M = 0.043316, Erro_R = 0.108552 Parâmetros: {'nI': 25, 'nR': 14, 'nO': 18, 'N1': 0.1, 'N2': 1e-06, 'N3': 0.001, 'τ': 20, 'mode': 'ahead'}\
Erro_M = 0.011625, Erro_R = 0.105866 Parâmetros: {'nI': 11, 'nR': 37, 'nO': 20, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.0001, 'τ': 11, 'mode': 'ahead'}\
 Erro_M = 0.022981, Erro_R = 0.076113 Parâmetros: {'nI': 21, 'nR': 23, 'nO': 18, 'N1': 0.1, 'N2': 0.0001, 'N3': 1e-05, 'τ': 12, 'mode': 'ahead'}






In [185]:
#params= best_trials[-12].params
params =  {'nI': 2, 'nR': 2, 'nO': 2, 'N1': 0.1, 'N2': 0.01, 'N3': 0.1, 'τ': 20, 'mode': 'past'}
params =  {'nI': 2, 'nR': 2, 'nO': 2, 'N1': 0.1, 'N2': 0.001, 'N3': 0.1, 'τ': 21, 'mode': 'past'}
params = list(params.values())
vec=[]


In [186]:
nI,nR,nO,N1,N2,N3,τ,mode= params
X,Y = PrepareData(sig,nI,nO,mode)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
print(rnn.BS)
#rnn.act='relu'
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    rnn.PredRulIntr2(x=X[i],maxRul=len(sig),lim=0.75,store=True,show=False)
    #rnn.PredRul(X[i],maxRul=len(sig),lim=0.75,store=True)
    rnn.fit(X[i],Y[i],store=True,start=0,show=False)
print(rnn.εM,rnn.εR)  
PlotPredErrorPLY(rnn,w=800,h=400)

[[ 0.11485451  0.7297006 ]
 [ 0.78482112 -0.92578497]]
0.03316346319830279 0.3415018256807431


In [147]:
print(rnn.wR)
print(rnn.wI)
print(rnn.wO)

[[-0.59302479 -1.09310716]
 [ 0.44948639 -1.10096431]]
[[0.30569114 1.71655044]
 [0.99933717 0.67232442]]
[[1.09748503 0.31061126]
 [1.32083593 0.13876695]]


In [ ]:
wR_hist = np.array(rnn.wR_hist).T
wI_hist = np.array(rnn.wI_hist).T
wO_hist = np.array(rnn.wO_hist).T

PlotSeriesPLY(ySeries=wR_hist)
PlotSeriesPLY(ySeries=wI_hist)
PlotSeriesPLY(ySeries=wO_hist)

In [37]:
i=int((len(sig)-nI)/2)
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.25
r_mU = np.mean(rnn.rulR[:i])*1.25
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

i=50
f=90
r_m = np.mean(rnn.rulR[i:f])
r_mL = np.mean(rnn.rulR[i:f])*0.3
r_mU = np.mean(rnn.rulR[i:f])*1.5
p_m = (np.mean(rnn.rulP[i:f]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

lower: 20.125 mid: 80.5 upper: 100.625
pred: 92.35185185185185
lower: 11.25 mid: 37.5 upper: 56.25
pred: 100.0


In [209]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ,mode= params
ηS = [N1,N2,N3]
X,Y = PrepareData(sig,nI,nO,mode)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
rnn.ref = len(sig)-nI

i=0

In [248]:
#rnn.PredRul(x=X[i],store=True)
print('iter',i+5)
rnn.PredRulIntr2(x=X[i],maxRul=len(sig),lim=0.75,store=True,show=True)
rnn.fit(X[i],Y[i],store=True,start=0,show=False)
i=i+1

iter 43
1 [0.98317837 0.88926475 0.82993636] [0.90984167 0.82240925 0.76839056] [0.98317837 0.88926475 0.82993636]
2 [0.99246187 0.90429507 0.8361331 ] [0.91704826 0.83512121 0.77346301] [0.99246187 0.90429507 0.8361331 ]
3 [0.99954123 0.91912341 0.84048909] [0.92068703 0.84749696 0.77557873] [0.99954123 0.91912341 0.84048909]
4 [1.00739891 0.9332492  0.84552358] [0.92603045 0.85902135 0.77911007] [1.00739891 0.9332492  0.84552358]
5 [1.01477713 0.94711232 0.85014244] [0.93100221 0.87034297 0.78230152] [1.01477713 0.94711232 0.85014244]
6 [1.02173976 0.96070341 0.85440289] [0.93569481 0.88145    0.7852339 ] [1.02173976 0.96070341 0.85440289]
7 [1.0282781  0.9740308  0.85829962] [0.94008672 0.89235147 0.78789167] [1.0282781  0.9740308  0.85829962]
8 [1.03440416 0.98709604 0.86184527] [0.94418604 0.90304764 0.79028428] [1.03440416 0.98709604 0.86184527]
9 [1.040129   0.9999012  0.86505145] [0.94799937 0.91353928 0.79241978] [1.040129   0.9999012  0.86505145]
10 [1.04546427 1.01244843 0.8